> **ANNEX.** Hardware smoke test (does Qwen fit on the 16 GB Mac mini?). Supports section 7 of notebook 01.
>
> Main reading: `01_FINAL_full_story` · engine: `02_deterministic_engine` · map: `00_INDEX`.

# 01 · Hardware: smoke test (memory and speed)
Goal: measure **memory and speed** of Qwen3.5-4B on this Mac mini (16 GB) before training.
Decision log: `finetune/FINETUNE_LOG.md`. None of this touches `validator/` or uses the evaluation receipts.

In [1]:
import platform, subprocess, mlx.core as mx
print(platform.platform())
print('Metal available:', mx.metal.is_available())
print(subprocess.run(['sysctl','-n','hw.memsize'],capture_output=True,text=True).stdout.strip(), 'bytes of RAM')

macOS-26.5.2-arm64-arm-64bit
Metal available: True
17179869184 bytes of RAM


## 1. Load the base model (not fine-tuned)
`mlx-community/Qwen3.5-4B-bf16` (≈8 GB). If it is not downloaded, `hf download` fetches it.

In [2]:
import time
from mlx_lm import load, generate
MODEL='mlx-community/Qwen3.5-4B-bf16'
t=time.time(); model,tok=load(MODEL); print('load: %.1f s'%(time.time()-t))
print('active memory GB: %.2f'%(mx.get_active_memory()/1e9))

/Users/geboou/finetune_project/.venv-ft/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 1469.20it/s]

load: 4.9 s
active memory GB: 8.41


## 2. A test generation
A realistic receipt-style line; it only measures speed, not quality. No reasoning (thinking off).

In [3]:
msgs=[{'role':'user','content':'Classify this expense line into one category. Line: "Kingfisher beer 650ml". Answer with the category only.'}]
prompt=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)
t=time.time(); out=generate(model,tok,prompt=prompt,max_tokens=40,verbose=True); dt=time.time()-t
print(repr(out)); print('peak memory GB: %.2f'%(mx.get_peak_memory()/1e9))

Al

cohol


Prompt: 40 tokens, 31.267 tokens-per-sec
Generation: 3 tokens, 17.219 tokens-per-sec
Peak memory: 8.531 GB
'Alcohol'
peak memory GB: 8.53


## Next
- Gate 0b: 20 LoRA iterations to measure tokens/s and training memory.
- Gate 1: untuned Qwen against the held-out set and the 32 receipts (baseline).
- Then: synthetic data, GRPO with LoRA, evaluation (see the log).